# Creating Star Schema 

This notebook explores the creation of a Star Schema from raw data using Apache Spark.

The data used in this notebook is from the Brazilian Basic Education Census, which is available at the [Brazilian government's open data portal](https://download.inep.gov.br/dados_abertos)

## Creating SparkSession

The first step is to create a SparkSession.

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import psycopg2
from datetime import datetime
from itertools import chain

In [4]:
spark = SparkSession.builder\
        .appName("CensoEscolarStarSchema")\
        .config("spark.driver.memory", "2g")\
        .config("spark.sql.shuffle.partitions", "2")\
        .getOrCreate()

## Transforming CSV to Parquet

This transformation aims to increase the speed of data loading by using Parquet files.

In [6]:


manual_schema = StructType([
    # Time Dimension
    StructField("NU_ANO_CENSO", IntegerType(), True),

    # Location Dimensions
    StructField("NO_UF", StringType(), True),
    StructField("SG_UF", StringType(), True),
    StructField("CO_UF", StringType(), True),
    StructField("NO_MUNICIPIO", StringType(), True),
    StructField("CO_MUNICIPIO", StringType(), True),

    # Infrastructure Dimensions (TP and IN columns)
    StructField("TP_DEPENDENCIA", IntegerType(), True),
    StructField("TP_LOCALIZACAO", IntegerType(), True),
    StructField("IN_AGUA_POTAVEL", IntegerType(), True),
    StructField("IN_ENERGIA_INEXISTENTE", IntegerType(), True),
    StructField("IN_ESGOTO_INEXISTENTE", IntegerType(), True),
    StructField("IN_BANHEIRO", IntegerType(), True),
    StructField("IN_BIBLIOTECA", IntegerType(), True),
    StructField("IN_REFEITORIO", IntegerType(), True),
    StructField("IN_COMPUTADOR", IntegerType(), True),
    StructField("IN_INTERNET", IntegerType(), True),
    StructField("IN_EQUIP_NENHUM", IntegerType(), True),

    # Metrics (Quantitative Counts)
    StructField("QT_DOC_BAS", IntegerType(), True),
    StructField("QT_DOC_INF", IntegerType(), True),
    StructField("QT_DOC_FUND", IntegerType(), True),
    StructField("QT_DOC_MED", IntegerType(), True),
    StructField("QT_MAT_BAS", IntegerType(), True),
    StructField("QT_MAT_INF", IntegerType(), True),
    StructField("QT_MAT_FUND", IntegerType(), True),
    StructField("QT_MAT_MED", IntegerType(), True),
    StructField("QT_MAT_BAS_ND", IntegerType(), True),
    StructField("QT_MAT_BAS_BRANCA", IntegerType(), True),
    StructField("QT_MAT_BAS_PRETA", IntegerType(), True),
    StructField("QT_MAT_BAS_PARDA", IntegerType(), True),
    StructField("QT_MAT_BAS_AMARELA", IntegerType(), True),
    StructField("QT_MAT_BAS_INDIGENA", IntegerType(), True)
])



In [8]:
data_csv = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(manual_schema) # Use the schema from documentation.txt[cite: 2]
    .option("delimiter", ";")
    .option("encoding", "latin1")
    .load("./data/*.csv")
    .limit(5000) # Start with 5,000 rows to ensure it finishes
)

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\Users\LENOVO\anaconda3\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\anaconda3\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\LENOVO\anaconda3\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\anaconda3\Lib\site-packages\py4j\clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error whi

Py4JError: An error occurred while calling o39.load

## data_csv = (
    spark
    .read
    .format("csv")
    .option("header", "true")
    .schema(manual_schema)          # <--- Use the manual schema here
    .option("inferSchema", "false") # <--- MUST be false to speed up
    .option("delimiter", ";")
    .option("encoding", "latin1")
    .load("./data/*.csv")
    .limit(50000)  # <--- Add this! It only takes 50k rows.
)

Write to Parquet file

In [ ]:
data_csv.write.parquet("./data/censo_escolar.parquet")

Reading from Parquet file

In [ ]:
data = (
    spark
    .read
    .format("parquet")
    .load("./data/censo_escolar.parquet/")
)

## Dimensions

The dimensions are...

---

The code below creates the dimensions based on a configuration dict.

```json
{
    "DIM_NAME":{
        # The fields are the table columns
        "fields":[
            {
                "field":"FIELD_1_NAME", # The column name
                "type":"FIELD_1_TYPE",  # The column type in spark
            },
            {
                "field":"FIELD_2_NAME",
                "type":"FIELD_2_TYPE",
            },
            ...
        ],
    }
}
```

In [ ]:
INTEGER_DIMENSIONS = [
    "TP_DEPENDENCIA",
    "TP_LOCALIZACAO",
    "IN_AGUA_POTAVEL",
    "IN_ENERGIA_INEXISTENTE",
    "IN_ESGOTO_INEXISTENTE",
    "IN_BANHEIRO",
    "IN_BIBLIOTECA",
    "IN_REFEITORIO",
    "IN_COMPUTADOR",
    "IN_INTERNET",
    "IN_EQUIP_NENHUM"
]

DIMENSION_TABLES_CONFIG = {
    "DIM_LOCAL":{
        "fields": [
            {"field":"NO_UF", "type":"string",},
            {"field":"SG_UF", "type":"string",},
            {"field":"CO_UF", "type":"string",},
            {"field":"NO_MUNICIPIO", "type":"string",},
            {"field":"CO_MUNICIPIO", "type":"string",}
        ]
    },
}

DIMENSION_TABLES_CONFIG.update(
    {
        "DIM_"+dimension.upper():{
            "fields": [
                {"field":dimension, "type":"integer"} 
            ]
        }
        for dimension in INTEGER_DIMENSIONS
    }
)

### Creating dimensions table in Postgres
-----------------------------------------------------------------------------------------------------------------------

Defining the properties of the Postgres Connection

In [ ]:
POSTGRES_USER = "postgres"
POSTGRES_PASSWORD = "admin123"
POSTGRES_DB = "censo_escolar"

# Used to connect to the PostgreSQL database server
# in spark session
POSTGRES_CONFIG = {
    "url":f"jdbc:postgresql://localhost:5432/{POSTGRES_DB}",
    "properties":{
        "user":POSTGRES_USER, 
        "password":POSTGRES_PASSWORD,
        "driver":"org.postgresql.Driver",
    },
}

Establishing connection to Postgres

In [ ]:
conn = psycopg2.connect(
    host="localhost",
    port="5432",

    dbname=POSTGRES_DB,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD
)

Function to create a Dimension table in Postgres using the configuration in DIMENSION_TABLES_CONFIG and adding an id column

The code below creates the dimensions
Spark will create a table with the name of the dimension and the columns in the configuration

In [ ]:
# Write data to Postgres
# Using the configuration in DIMENSION_TABLES_CONFIG
# With id as the primary key

for table_name, table_config in DIMENSION_TABLES_CONFIG.items():
    
    print(f"[{datetime.now()}] Writing {table_name}")
    
    data\
    .select(
        [
            F
            .col(field["field"])
            .cast(field["type"])
            .alias(field["field"])
            
            for field
            in table_config["fields"]
        ]
    )\
    .distinct()\
    .withColumn(
        "id", F.monotonically_increasing_id()
    )\
    .write\
    .jdbc(
        **POSTGRES_CONFIG,
        table=table_name,
        mode="overwrite"
    )
    
    print(f"[{datetime.now()}] Wrote {table_name}")
    # Define id as the primary key
    cursor = conn.cursor()
    cursor.execute(
        f"ALTER TABLE {table_name} ADD PRIMARY KEY (id);"
    )
    cursor.close()
    conn.commit()

    print(f"[{datetime.now()}] Added primary key to {table_name}")
    print(f"[{datetime.now()}] Done")

## Facts table

The definition of the facts table follows a different pattern than the dimensions.

The table schema is previously defined to properly define the foreing keys.


Defining the facts table schema
Metrics + Facts + Dimensions (Foreign Keys)

In [ ]:
FACT_TABLE_NAME = "FACT_CENSO_ESCOLAR"

FACT_COLUMNS = [
    "QT_DOC_BAS",	# Número de Docentes da Educação Básica
    "QT_DOC_INF",	# Número de Docentes da Educação Infantil
    "QT_DOC_FUND",	# Número de Docentes do Ensino Fundamental
    "QT_DOC_MED",	# Número de Docentes do Ensino Médio

    "QT_MAT_BAS",	# Número de Matrículas na Educação Básica (TOTAL)
    "QT_MAT_INF",	# Número de Matrículas na Educação Infantil
    "QT_MAT_FUND",	# Número de Matrículas no Ensino Fundamental
    "QT_MAT_MED",	# Número de Matrículas no Ensino Médio

    "QT_MAT_BAS_ND",	    # Número de Matrículas na Educação Básica - Cor/Raça Não Declarada
    "QT_MAT_BAS_BRANCA",	# Número de Matrículas na Educação Básica - Cor/Raça Branca
    "QT_MAT_BAS_PRETA",	    # Número de Matrículas na Educação Básica - Cor/Raça Preta
    "QT_MAT_BAS_PARDA",	    # Número de Matrículas na Educação Básica - Cor/Raça Parda
    "QT_MAT_BAS_AMARELA",	# Número de Matrículas na Educação Básica - Cor/Raça Amarela
    "QT_MAT_BAS_INDIGENA",	# Número de Matrículas na Educação Básica - Cor/Raça Indígena
    
    "NU_ANO_CENSO"
]

FACT_CONFIG = {
    fact:{
        "fields": [
            {"field":fact, "type":"integer"}
        ]
    }
    for fact in FACT_COLUMNS
}

DIMENSION_ID_CONFIG = {
    table_name:[
        field['field'] 
        for field 
        in table_fields['fields']
    ]
    for table_name, table_fields in DIMENSION_TABLES_CONFIG.items()
}

FACT_TABLE_ALL_COLUMNS_ORDERED = FACT_COLUMNS + list(map(lambda col:"ID_"+col, DIMENSION_ID_CONFIG.keys()))

Before inserting the data into the facts table, we need to create a function to create the facts table in Postgres

The code below creates the facts table in Postgres using the configuration in FACT_TABLES_CONFIG and adding an id column for each dimension

In [ ]:
# Create fact table
# Using the configuration in FACT_CONFIG
# With id as the primary key

# Avoid inserting a backslash into a f-string
comma_break_line = ",\n\t\t\t"
facts_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {FACT_TABLE_NAME} (
        id SERIAL PRIMARY KEY,
        { 
            comma_break_line.join(
                [
                    f"{field} INTEGER" 
                    for field in FACT_COLUMNS
                ]
                +[
                    f"ID_{dim_table} BIGINT"
                    for dim_table in DIMENSION_ID_CONFIG.keys()
                ]
            )
        }
    );
    
    -- Adding Foreign Keys
    ALTER TABLE {FACT_TABLE_NAME}
    {
        comma_break_line.join(
            [
                f"ADD CONSTRAINT {FACT_TABLE_NAME}_{dim_table}_fk FOREIGN KEY (ID_{dim_table}) REFERENCES {dim_table}(id)"
                for dim_table in DIMENSION_ID_CONFIG.keys()
            ]
        )
    }
"""

In [ ]:
print(facts_table_sql)

Executing the function to create the facts table in Postgres

In [ ]:
print(f"[{datetime.now()}] Creating facts table")

cursor = conn.cursor()
try:
    cursor.execute(facts_table_sql)
    cursor.close()
    conn.commit()
except Exception as e:
    print(e)
    conn.rollback()
    cursor.close()
else:
    print(f"[{datetime.now()}] Created facts table")
    print(f"[{datetime.now()}] Done")


Extracting the data

In [ ]:
facts_data = data\
    .select(
        [
            *chain(
                *DIMENSION_ID_CONFIG.values(), 
                FACT_CONFIG.keys()
            )
        ]
    )

Adding the ids for each dimension

In [ ]:
# Joining the id of the dimensions

for table_name, table_fields in DIMENSION_ID_CONFIG.items():
    
    # Read the dimension data from Postgres
    dim_table = spark.read\
        .jdbc(
            **POSTGRES_CONFIG,
            table=table_name,
        )\
        .withColumnRenamed("id", f"ID_{table_name}")
    
    # Join the dimension data with the fact data
    facts_data = facts_data\
        .join(
            dim_table,
            on=table_fields,
            how="left"
        )\
        .drop(*table_fields)

Saving data to Postgres

In [ ]:
# Order the columns to match the fact table on postgres
# and save the data
facts_data\
    .select(*FACT_TABLE_ALL_COLUMNS_ORDERED)\
    .write\
    .jdbc(
        **POSTGRES_CONFIG,
        table=FACT_TABLE_NAME,
        mode="append"
    )

## References

https://towardsdatascience.com/explaining-technical-stuff-in-a-non-techincal-way-apache-spark-274d6c9f70e9

https://towardsdatascience.com/adding-sequential-ids-to-a-spark-dataframe-fa0df5566ff6

https://sparkbyexamples.com/pyspark/pyspark-read-and-write-parquet-file/

https://www.psycopg.org/docs/usage.html
